# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [29]:
# imports

import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display

In [30]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2:latest'

In [31]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

openai = OpenAI()

ollama = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

API key looks good so far


In [19]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [ ]:
# Get gpt-4o-mini to answer, with streaming

def answer_question_gpt(question):
    system_prompt = """
    You are an assistant that answers questions. Try to help in a way possible.
    You respond with an explanation of the code.
    """

    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": question}] 

    # Step 3: Call OpenAI
    stream = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=messages,
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)


In [ ]:
print(answer_question_gpt(question))

The code you've provided is using a combination of a generator function and a set comprehension to yield unique authors from a collection of books. Here's a breakdown of what each part does:

1. **`{book.get("author") for book in books if book.get("author")}`**: This part is a set comprehension that creates a set of authors. Here's how it works:
   - `for book in books`: It iterates over an iterable named `books`, where each `book` is presumably a dictionary containing data about each book.
   - `book.get("author")`: This attempts to retrieve the value associated with the key `"author"` from the `book` dictionary. The `get` method is used because it returns `None` if the key does not exist, instead of raising a `KeyError`.
   - `if book.get("author")`: This condition filters the authors, effectively ensuring that only non-empty (truthy) author values are included in the resulting set.
   - The braces `{}` around the comprehension create a set, which inherently ensures that all authors are unique (i.e., no duplicates).

2. **`yield from ...`**: The `yield from` statement is used within a generator function to yield all values from the iterable that follows it. In this case, it yields each unique author from the set created by the set comprehension. 

A generator function is a type of function that returns an iterator—one that we can iterate over (one value at a time), rather than computing and returning all values at once.

### Overall Functionality:
- The overall effect of the code is to produce a generator that will yield each unique author from the `books` collection, skipping any entries that do not have an author. 

### Example:
If `books` were as follows:
python
books = [
    {"title": "Book 1", "author": "Author A"},
    {"title": "Book 2", "author": "Author B"},
    {"title": "Book 3", "author": "Author A"},  # Duplicate
    {"title": "Book 4"}  # No author
]

The code would yield:

"Author A"
"Author B"


### Why Use This Code?
- It simplifies the process of filtering and collecting unique authors from a list of books with minimal lines of code while ensuring that the values returned do not include duplicates or `None` entries.

None


In [32]:
# Get Llama 3.2 to answer

def answer_question_llama(question):
    system_prompt = """
    You are an assistant that answers questions. Try to help in a way possible.
    You respond with an explanation of the code.
    """

    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": question}] 

    # Step 3: Call OpenAI
    stream = ollama.chat.completions.create(
        model=MODEL_LLAMA,
        messages=messages,
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)


In [33]:
print(answer_question_llama(question))

**Generator Expression Explanation**

This code is using a combination of Python features to extract specific data from a list of dictionaries. Let's break it down:

1. `yield from`: This is a syntax feature introduced in Python 3.3, which allows you to use the `yield` keyword not just within functions (where it typically resides), but also inside an expression.
2. `{... for book in books if ...}`: This is called a generator expression. It's similar to a list comprehension, but instead of evaluating everything immediately, it generates values on-the-fly as needed.

The entire expression `yield from {book.get("author") for book in books if book.get("author")}`:

* Iterates over the `books` collection (assuming it's an iterable).
* For each dictionary-like object (`book`) in the `books` collection:
	+ Checks if the value associated with key `"author"` exists using `book.get("author")`. If it doesn't exist, the iteration is skipped.
	+ If the key exists, its corresponding value (`book.get("author")`) is extracted and yielded (i.e., added to) a dictionary-like object containing only this extracted value.

**Why?**

This expression serves several purposes:

* **Lazy evaluation**: Instead of building an intermediate data structure (e.g., a list), it generates values as needed, which can be useful when working with large datasets.
* **Filtering**: By adding the `if` condition (`book.get("author")` is truthy), you're effectively filtering out dictionaries that don't have an `"author"` key.
* **Efficient iteration**: The use of a generator expression allows Python to avoid creating unnecessary intermediate collections or storing data in memory.

In summary, this code takes advantage of generator expressions and the `get()` method to lazily extract author names from a list of dictionaries while filtering out dictionaries without an `"author"` key.

None
